<a href="https://colab.research.google.com/github/bosywahab818-a11y/flyrank-ml-internship-bouthina/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bosywahab818-a11y/flyrank-ml-internship-bouthina/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [5]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [7]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│  78835655 │ 2025-01-27 │ 2026-06-30 │
└───────────┴────────────┴────────────┘



In [9]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 1
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [10]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘



One row represents the daily performance of one content item for one client.

For this assignment, I use the March 2026 window (2026-03-01 to 2026-03-31) as a mid-panel development month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Features:
- march_impressions
- march_clicks
- march_avg_position
- march_pageviews
- march_sessions

Label / proxy:
- april_impressions, representing the future April performance outcome.

Context:
- report_date
- client_hash_id
- content_hash_id
- month

Excluded:
- gsc_data_available and ga4_data_available are used to check data availability, not as model features.
- april_impressions is excluded from the honest feature set because it is only known after the March decision moment.
- Any future-period or label-derived field is excluded to prevent data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").show()

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [12]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [13]:
con.sql(f"""
SELECT
    COUNT(*) AS duplicate_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
) AS duplicates
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┐
│ duplicate_groups │
│      int64       │
├──────────────────┤
│                0 │
└──────────────────┘



In [14]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) AS impressions_available,
    COUNT(*) FILTER (WHERE gsc_clicks IS NOT NULL) AS clicks_available,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NOT NULL) AS position_available,
    COUNT(*) FILTER (WHERE ga4_pageviews IS NOT NULL) AS pageviews_available,
    COUNT(*) FILTER (WHERE ga4_sessions IS NOT NULL) AS sessions_available
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────────────┬──────────────────┬────────────────────┬─────────────────────┬────────────────────┐
│  rows   │ impressions_available │ clicks_available │ position_available │ pageviews_available │ sessions_available │
│  int64  │         int64         │      int64       │       int64        │        int64        │       int64        │
├─────────┼───────────────────────┼──────────────────┼────────────────────┼─────────────────────┼────────────────────┤
│ 9841378 │               9841378 │          9841378 │            3611061 │             6822637 │            6822637 │
└─────────┴───────────────────────┴──────────────────┴────────────────────┴─────────────────────┴────────────────────┘



In [15]:
con.sql(f"""
DESCRIBE SELECT *
FROM {TABLES['fact_daily']}
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [16]:
con.sql(f"""
SELECT
    COUNT(*) AS client_count,
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start,
    MIN(ga4_data_start) AS earliest_ga4_start,
    MAX(ga4_data_start) AS latest_ga4_start
FROM {TABLES['dim_clients']}
""").show()

┌──────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┐
│ client_count │ earliest_gsc_start │ latest_gsc_start │ earliest_ga4_start │ latest_ga4_start │
│    int64     │        date        │       date       │        date        │       date       │
├──────────────┼────────────────────┼──────────────────┼────────────────────┼──────────────────┤
│          104 │ 2025-01-27         │ 2026-06-02       │ 2025-10-29         │ 2026-06-01       │
└──────────────┴────────────────────┴──────────────────┴────────────────────┴──────────────────┘



In [17]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_query_90d']}
LIMIT 1
""").show()

┌─────────────────────────┬──────────────────────────┬────────────────────────┬──────────────────┬───────────────────┬──────────────┬────────────┬─────────────────┬────────────┬────────────────────┬───────────────┬────────────────────┬───────────────┬────────────────────┬─────────────────────┬─────────────────────┬───────────────────────────────┬─────────────────────────────┬──────────────────┬────────────────────────┬──────────────────────────────┐
│     client_hash_id      │     content_hash_id      │     query_hash_id      │ query_char_count │ query_token_count │ window_start │ window_end │ impressions_90d │ clicks_90d │ impressions_last30 │ clicks_last30 │ impressions_prev30 │ clicks_prev30 │  avg_position_90d  │ avg_position_last30 │ avg_position_prev30 │ content_total_impressions_90d │ content_visible_query_count │ rare_query_count │ rare_impressions_share │ anonymized_impressions_share │
│         varchar         │         varchar          │        varchar         │      int64  

In [18]:
con.sql(f"""
SELECT
    COUNT(*) AS rows_after_march,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'
""").show()

┌──────────────────┬────────────┬────────────┐
│ rows_after_march │ first_date │ last_date  │
│      int64       │    date    │    date    │
├──────────────────┼────────────┼────────────┤
│         10424730 │ 2026-04-01 │ 2026-04-30 │
└──────────────────┴────────────┴────────────┘



In [19]:
con.sql(f"""
SELECT
    COUNT(*) AS march_pairs,
    COUNT(*) FILTER (WHERE april.impressions > 0) AS pairs_with_april_impressions
FROM (
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
) AS march
LEFT JOIN (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
) AS april
ON march.client_hash_id = april.client_hash_id
AND march.content_hash_id = april.content_hash_id
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬──────────────────────────────┐
│ march_pairs │ pairs_with_april_impressions │
│    int64    │            int64             │
├─────────────┼──────────────────────────────┤
│      331437 │                       176441 │
└─────────────┴──────────────────────────────┘



In [20]:
feature_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) FILTER (
            WHERE gsc_avg_position > 0
        ) AS march_avg_position,
        SUM(ga4_pageviews) AS march_pageviews,
        SUM(ga4_sessions) AS march_sessions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY
        client_hash_id,
        content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    march.client_hash_id,
    march.content_hash_id,
    march.march_impressions,
    march.march_clicks,
    march.march_avg_position,
    march.march_pageviews,
    march.march_sessions,
    april.april_impressions
FROM march
LEFT JOIN april
    ON march.client_hash_id = april.client_hash_id
    AND march.content_hash_id = april.content_hash_id
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_pageviews,march_sessions,april_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,1.0,6787.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,3.307255,0.0,0.0,405.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,3.0,8475.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,2.0,6091.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,23.314103,8.0,7.0,67.0


In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

leaky_frame = feature_frame.dropna(subset=["april_impressions"]).copy()

X = leaky_frame[[
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "april_impressions"   # DELIBERATE LEAK
]]

y = leaky_frame["april_impressions"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leaky R²:", r2_score(y_test, pred))

Leaky R²: 0.9999889697989749


In [22]:
honest_frame = feature_frame.dropna(subset=["april_impressions"]).copy()

X = honest_frame[[
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_pageviews",
    "march_sessions"
]]

y = honest_frame["april_impressions"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

honest_model = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

print("Honest R²:", r2_score(y_test, honest_pred))

Honest R²: 0.7378266008529444


March rows: 9,841,378
First date: 2026-03-01
Last date: 2026-03-31

GSC available: 3,611,061
GA4 available: 413,966

Duplicate groups: 0

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This slice has several limitations.

First, client histories are unbalanced: GSC and GA4 data do not start on the same date for every client.

Second, GA4 is not available for every row. In March 2026, only 413,966 of 9,841,378 rows had ga4_data_available = TRUE, so GA4-based features may cover only part of the data.

Third, the April proxy is a future outcome and is not available at the March decision moment. It must never be used as a feature.

Finally, this analysis uses March 2026 for development and April 2026 as the future outcome window. The June 2026 sample should remain sealed and should not be used to develop the label logic.

In [23]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").show()

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.